Get occurrences in Darwin Core format.

Fetch all inferences for this combination of species, site and year.
Returns: list of dict, one Darwin Core occurrence record per inference
row (see _row_to_dwc), or None if there are no results.


## Connect to database
Follow the instruction in the Notebook [00_Setup_DB_Connection](./00_Setup_DB_Connection.ipynb).

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Load the database username and password
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Read the username and password
username = os.getenv("DB_USERNAME")
password = os.getenv("DB_PASSWORD")

DB_CONFIG = {
    'host': 'localhost',
    'port': 8432,  # the port opened using the ssh tunnel
    'user': username,
    'password': password,
    'database': 'ecomon'
}

instance_name = "validate"
model_name = 'BNP_v3_0_glob11k_prev3_cw5s_sz2s'

## Test the connection

In [4]:
from bslquery import BioSoundsLabDwC

# Initialize the database helper
db = BioSoundsLabDwC(DB_CONFIG, verbose=True)

# Test the connection
db.test_connection()

Connection successful!


True

## Fetch sites and years
This will return one entry for each combination of site prefix and year, and the count of records.

In [7]:
# List all sites and years combinations
site_years = db.sites_years()
site_years

[{'prefix': 'AKWAMO9801G',
  'name': 'Britz, BBBRI_9801G',
  'year': 2024,
  'count': 19986},
 {'prefix': 'AKWAMO0609A',
  'name': 'Krofdorf, HEKRO_0609A',
  'year': 2025,
  'count': 30812},
 {'prefix': 'AKWAMO1205H',
  'name': 'Neusorgefeld, BBNSF_1205H',
  'year': 2024,
  'count': 19298},
 {'prefix': 'AKWAMO12C1A',
  'name': 'Schorfheide, Totalreservat, BBSHT_12C1A',
  'year': 2023,
  'count': 30592},
 {'prefix': 'AKWAMO03C1A',
  'name': 'Vogelherd, Naturwaldreservat, NIVOH_03C1A',
  'year': 2024,
  'count': 30813}]

## Fetch species labels
Just to make sure the names are in the database.

In [16]:
species = [
    'Falco naumanni',
    'Falco tinnunculus',
    'Falco vespertinus',
    'Falco eleonorae',
    'Falco columbarius',
    'Falco subbuteo',
    'Falco cherrug',
    'Falco rusticolus',
    'Falco peregrinus'
]
species_labels = []
for spc in species:
    species_labels.append({
        'label_id': db.fetch_label_id_by_species_name(spc),
        'ScientificName_IOC10.1': spc
    })
species_labels

[{'label_id': 6495, 'ScientificName_IOC10.1': 'Falco naumanni'},
 {'label_id': 6508, 'ScientificName_IOC10.1': 'Falco tinnunculus'},
 {'label_id': 6509, 'ScientificName_IOC10.1': 'Falco vespertinus'},
 {'label_id': 6487, 'ScientificName_IOC10.1': 'Falco eleonorae'},
 {'label_id': 6482, 'ScientificName_IOC10.1': 'Falco columbarius'},
 {'label_id': 6506, 'ScientificName_IOC10.1': 'Falco subbuteo'},
 {'label_id': 6480, 'ScientificName_IOC10.1': 'Falco cherrug'},
 {'label_id': 6503, 'ScientificName_IOC10.1': 'Falco rusticolus'},
 {'label_id': 6498, 'ScientificName_IOC10.1': 'Falco peregrinus'}]

## Fetch model id
Just to make sure the model is in the database.

In [17]:
model_id = db.fetch_model_id_by_model_name(model_name)
model_id

11

## Fetch inferences

___Before executing the next cell, create a folder where the spreadsheets with the data will be stored.___

The folder should be called: examples/data (this folder will not be committed).

In [21]:
import json

def save_output_data(rows, species_name, instance_name):
    # export as json
    filename = f"Inf_{species_name}_{instance_name}.json".replace(' ', '_')
    with open(f"data/{filename}", "w") as f:
        json.dump(rows, f, indent=2)

In [22]:
for species_current in species_labels:
    species_name = species_current['ScientificName_IOC10.1']
    inferences = list()
    for current_site_year in site_years:
        new = db.fetch_inferences(species_name, model_name, current_site_year['prefix'], current_site_year['year'])
        if new is not None:
            inferences += new
    save_output_data(inferences, species_name, instance_name)
print("Done")

Processing: AKWAMO9801G (8), 2024, Falco naumanni (6495), BNP_v3_0_glob11k_prev3_cw5s_sz2s (11)
Found no results
Processing: AKWAMO0609A (29), 2025, Falco naumanni (6495), BNP_v3_0_glob11k_prev3_cw5s_sz2s (11)
Found 4 results
Processing: AKWAMO1205H (23), 2024, Falco naumanni (6495), BNP_v3_0_glob11k_prev3_cw5s_sz2s (11)
Found 2 results
Processing: AKWAMO12C1A (15), 2023, Falco naumanni (6495), BNP_v3_0_glob11k_prev3_cw5s_sz2s (11)
Found 14 results
Processing: AKWAMO03C1A (25), 2024, Falco naumanni (6495), BNP_v3_0_glob11k_prev3_cw5s_sz2s (11)
Found 11 results
Processing: AKWAMO9801G (8), 2024, Falco tinnunculus (6508), BNP_v3_0_glob11k_prev3_cw5s_sz2s (11)
Found 372 results
Processing: AKWAMO0609A (29), 2025, Falco tinnunculus (6508), BNP_v3_0_glob11k_prev3_cw5s_sz2s (11)
Found 308 results
Processing: AKWAMO1205H (23), 2024, Falco tinnunculus (6508), BNP_v3_0_glob11k_prev3_cw5s_sz2s (11)
Found 174 results
Processing: AKWAMO12C1A (15), 2023, Falco tinnunculus (6508), BNP_v3_0_glob11k_p

## Validate

In [25]:
from bslquery.DwCValidator import DwCValidator

validator = DwCValidator()
validator.validate(inferences)

True